IMPORT

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

DEFINE PATH

In [0]:
raw_path = "abfss://source@azuresalesordertest.dfs.core.windows.net/bigmart/source"

silver_table_path = "abfss://source@azuresalesordertest.dfs.core.windows.net/bigmart/silver/"

checkpoint_path = "abfss://source@azuresalesordertest.dfs.core.windows.net/bigmart/checkpoint/"

DEFINE SCHEMA

In [0]:
my_struct_schema = StructType([
    StructField("Item_Identifier", StringType(), True),
    StructField("Item_Weight", DoubleType(), True),
    StructField("Item_Fat_Content", StringType(), True),
    StructField("Item_Type", StringType(), True),
    StructField("Item_Visibility", DoubleType(), True),
    StructField("Item_MRP", DoubleType(), True),
    StructField("Outlet_Identifier", StringType(), True),
    StructField("Outlet_Establishment_Year", StringType(), True),
    StructField("Outlet_Size", StringType(), True),
    StructField("Outlet_Location_Type", StringType(), True),
    StructField("Outlet_Type", StringType(), True),
    StructField("Item_Outlet_Sales", DoubleType(), True)
])

Read Incremental Daily Files
Auto Loader

In [0]:
df_raw = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .schema(my_struct_schema) \
    .load(raw_path)

Silver Transformations
Keep Full History

In [0]:

df_silver = df_raw \
    .withColumn(
        "Item_Weight",
        col("Item_Weight").cast("double")
    ) \
    .withColumn(
        "Item_Visibility",
        col("Item_Visibility").cast("double")
    ) \
    .withColumn(
        "Item_MRP",
        col("Item_MRP").cast("double")
    ) \
    .withColumn(
        "Item_Outlet_Sales",
        col("Item_Outlet_Sales").cast("double")
    ) \
    .withColumn(
        "Outlet_Establishment_Year",
        col("Outlet_Establishment_Year").cast("int")
    ) \
    .withColumn(
        "Item_Fat_Content",
        regexp_replace(
            col("Item_Fat_Content"),
            "Regular",
            "Reg"
        )
    ) \
    .withColumn(
        "Item_Fat_Content",
        regexp_replace(
            col("Item_Fat_Content"),
            "Low Fat",
            "LowFat"
        )
    ) \
    .fillna({
        "Item_Weight": 0.0,
        "Item_Visibility": 0.0,
        "Outlet_Size": "Unknown"
    }) \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

Write Silver Delta Table Append Only

In [0]:
query = df_silver.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        checkpoint_path
    ) \
    .trigger(availableNow=True) \
    .start(silver_table_path)

query.awaitTermination()


Register Silver Table

In [0]:

spark.sql(f"""

CREATE TABLE IF NOT EXISTS salesorder.bigmart.bigmart_sales

USING DELTA

LOCATION '{silver_table_path}'

""")

Validate Silver Table

In [0]:
display(
    spark.table("salesorder.bigmart.bigmart_sales")
)